# ASE relaxation script

This is a script for the relaxation of an atomic system using ase and Matlantis. This script can be run with minimal editing or be used as a base to develop you own scripts.

## Imports
These are the modules and functions that are needed for the script to work as intended. Additional imports can be added here if the script is modified. Remember to rerun this cell if any imports is added.

In [ ]:
from ase import io
from ase.optimize import FIRE, BFGS, MDMin, CellAwareBFGS
from ase.optimize.sciopt import SciPyFminCG
from ase.filters import ExpCellFilter, FrechetCellFilter

import pfp_api_client
from pfp_api_client.pfp.calculators.ase_calculator import ASECalculator
from pfp_api_client.pfp.estimator import Estimator, EstimatorCalcMode

## Matlantis calculator
Set the calculator that will be used to get the energy and forces for the relaxation. __Edit__ this cell to change the calculator mode (e.g. _PBE_, _PBE_PLUS_D3_, _R2SCAN_, _R2SCAN_PLUS_D3_). This will use the different versions of the potential fitted agains different DFT functionals.

In [ ]:
estimator = Estimator(calc_mode=EstimatorCalcMode.PBE_PLUS_D3, model_version='v9.0.0')
calculator = ASECalculator(estimator)

print(f'pfp_api_client: {pfp_api_client.__version__}')

## Parameters
Set these parameters to control the job (like a vasp INCAR). File paths should be absolute or relative to the directory where the notebook file is before job submission.
* __input_file__ is the geometry top be relaxed (i.e. POSCAR). Will use the last frame if file contains multiple frames, use ase @ syntax to select a specific frame.
* __output_file__ is where to save the relaxed geometry (i.e. CONTCAR). This file is only created when the relaxation finishes.
* __trajectory_file__ is where to save the geoemtry trajectory while relaxing (i.e. OUTCAR).
* __log_file__ is where to save the logging output for the relaxation. Use '-' to print to stdout (i.e. the notebook) and None to skip.
* __optimiser__ is the optimiser to use (i.e. IBRION or IOPT). Allowed options are FIRE, BFGS, MDMin, CellAwareBFGS and SciPyFminCG. My sugesstions are BFGS/SciPyFminCG when close to a minima or FIRE/MDMin when far from it.
* __relax_cell__ selects if the cell should be relaxed in adition to the coordinates (i.e. ISIF).
* __fmax__ set the force convergence for the relaxation in eV/Å (i.e EDIFFG)

If the script is run in interactive mode or by the _Run in Background_ button edit these parameters directly in this cell. If the script is run by _mtl-bg-job_ the parameters can be overwriten by adding _-p parameter-name parameter-value_ to the command.

In [ ]:
input_file  = '../ZrN_110/1023K/surface.traj'
output_file = '../ZrN_110/1023K/surface_relaxed.traj'
trajectory_file = '../ZrN_110/1023K/surface_relax_traj.traj'
log_file    = '-'
optimiser   = FIRE
relax_cell  = False   
fmax        = 0.05    

Parameters read from _mtl-bg-job_ might have the wrong type. This cell will parse them correctly.

In [ ]:
if isinstance(optimiser, str):
    optimiser = {'FIRE': FIRE, 'BFGS':BFGS, 'MDMin':MDMin, 'CellAwareBFGS':CellAwareBFGS, 'SciPyFminCG':SciPyFminCG}[optimiser]

if isinstance(relax_cell, str):
    relax_cell = (relax_cell == 'True')

fmax = float(fmax)

# Run relaxation
Reads the input file and sets the calculator for the atoms.

In [ ]:
atoms = io.read(input_file)
atoms.calc = calculator

In [ ]:
relax = FrechetCellFilter(atoms) if relax_cell else atoms

dyn = optimiser(relax,
                logfile=log_file, loginterval=1)

traj = io.Trajectory(trajectory_file, 'w', atoms)
dyn.attach(traj)

try:
    status = dyn.run(fmax=fmax)
except:
    status = False
    raise
finally:
    io.write(output_file, atoms)
    print(f'Optimisation ended with status {status}.')